# Graph-Enriched Context Provider

This notebook uses `Neo4jContextProvider` from the [`agent-framework-neo4j`](https://github.com/neo4j-labs/neo4j-maf-provider) package — the same provider from the previous notebook, but with an additional `retrieval_query` parameter that enables **graph enrichment**.

When configured with a `retrieval_query`, the provider uses `VectorCypherRetriever` internally — running vector search first to find matching nodes, then executing a custom Cypher query to traverse the graph and pull in related entities. Instead of just returning matched text chunks, the provider will traverse relationships to include company names, products, and risk factors.

***

Load the environment variables and import the required modules.

In [9]:
import sys
sys.path.insert(0, '../shared')

from azure.identity import DefaultAzureCredential
from azure.identity.aio import AzureCliCredential

from agent_framework.azure import AzureAIClient
from agent_framework_neo4j import (
    AzureAIEmbedder,
    AzureAISettings,
    Neo4jContextProvider,
    Neo4jSettings,
)

from config import get_agent_config

Get the configuration and create the embedder.

In [10]:
# Get configuration from environment
config = get_agent_config()
neo4j_settings = Neo4jSettings()
azure_settings = AzureAISettings()

# Create embedder
sync_credential = DefaultAzureCredential()

embedder = AzureAIEmbedder(
    endpoint=azure_settings.inference_endpoint,
    credential=sync_credential,
    model=azure_settings.embedding_model,
)

## Define the Retrieval Query

The retrieval query is a Cypher query that traverses the graph after vector search finds matching chunks. It receives `node` (the matched Chunk) and `score` (similarity score) from the vector search.

This query traverses:
- `Company` → `FROM_CHUNK` → `Chunk` (entities extracted from the chunk)
- `Company` → `FACES_RISK` → `RiskFactor` (risks the company faces)
- `Company` → `OFFERS` → `Product` (products the company offers)

In [11]:
# Graph-enriched retrieval query
# Appended after vector search: YIELD node, score
RETRIEVAL_QUERY = """
OPTIONAL MATCH (company:Company)-[:FROM_CHUNK]->(node)
OPTIONAL MATCH (company)-[:FACES_RISK]->(risk:RiskFactor)
WITH node, score, company,
     collect(DISTINCT risk.name)[0..5] AS risks
OPTIONAL MATCH (company)-[:OFFERS]->(product:Product)
WITH node, score, company, risks,
     collect(DISTINCT product.name)[0..5] AS products
RETURN
    node.text AS text,
    score,
    company.name AS company,
    company.ticker AS ticker,
    risks,
    products
ORDER BY score DESC
"""

print("Retrieval Query Pattern:")
print("  Company -[:FROM_CHUNK]-> Chunk (matched node)")
print("  Company -[:FACES_RISK]-> RiskFactor")
print("  Company -[:OFFERS]-> Product")

Retrieval Query Pattern:
  Company -[:FROM_CHUNK]-> Chunk (matched node)
  Company -[:FACES_RISK]-> RiskFactor
  Company -[:OFFERS]-> Product


> The retrieval query must use the `node` and `score` variables provided by the vector search. It must return at least `text` and `score` columns.

***

## Create the Graph-Enriched Context Provider

When a `retrieval_query` is provided, the provider internally uses `VectorCypherRetriever` instead of the basic `VectorRetriever`. Results include both the matched text and traversed graph context.

In [12]:
# Create context provider with graph-enriched mode
provider = Neo4jContextProvider(
    uri=neo4j_settings.uri,
    username=neo4j_settings.username,
    password=neo4j_settings.get_password(),
    index_name=neo4j_settings.vector_index_name,
    index_type="vector",
    retrieval_query=RETRIEVAL_QUERY,
    embedder=embedder,
    top_k=5,
    context_prompt=(
        "## Graph-Enriched Knowledge Context\n"
        "The following information combines semantic search results with "
        "graph traversal to provide company, product, and risk context:"
    ),
)

> The key difference from the previous notebook is the `retrieval_query` parameter. This tells the provider to use graph traversal for enriched results.

***

## Run the Agent with Graph-Enriched Context

The agent now receives context that includes company names, products, and risk factors alongside the matched text. The agent is created once and reused for all queries below.

In [13]:
# Connect provider and create agent
credential = AzureCliCredential()
await provider.__aenter__()
print("Connected to Neo4j with graph-enriched mode!\n")

client = AzureAIClient(
    project_endpoint=config.project_endpoint,
    model_deployment_name=config.model_name,
    credential=credential,
)

agent = client.as_agent(
    name="workshop-graph-enriched-agent",
    instructions=(
        "You are a helpful assistant that answers questions about companies "
        "using graph-enriched context. Your context includes:\n"
        "- Semantic search matches from company filings\n"
        "- Company names and ticker symbols\n"
        "- Products the company mentions\n"
        "- Risk factors the company faces\n\n"
        "When answering, cite the company, relevant products, and risks. "
        "Be specific and reference the enriched graph data."
    ),
    context_providers=[provider],
)

session = agent.create_session()

async def ask(query):
    """Run a query against the agent and print the response."""
    print(f"User: {query}\n")
    print("Assistant: ", end="", flush=True)
    response = await agent.run(query, session=session)
    print(response.text)
    print()

print(f"Agent '{agent.name}' created")

Connected to Neo4j with graph-enriched mode!

Agent 'workshop-graph-enriched-agent' created


In [14]:
await ask("What are Apple's main products and what risks does the company face?")

User: What are Apple's main products and what risks does the company face?

Assistant: Apple Inc. (AAPL) is known for a range of innovative products, with its main offerings including:

1. **iPhone**: One of the most popular smartphones globally that significantly contributes to Apple's revenue.
2. **iPad**: A line of tablets that targets various consumer segments, including education and professional markets.
3. **Mac**: Apple's personal computers and laptops, known for their design and performance.
4. **Apple Watch**: A smartwatch known for its health and fitness tracking capabilities.
5. **Apple TV**: A digital media player and streaming device.
6. **Services**: This includes the App Store, Apple Music, iCloud, and Apple TV+, which have become crucial revenue streams for the company.

### Risks Facing Apple:
1. **Component Supply Shortages**: Apple is vulnerable to shortages in the supply of critical components, which can disrupt product availability and affect sales.
   
2. **Depen

> Graph enrichment provides structured metadata (company, ticker, risks, products) alongside the text, enabling more specific and accurate responses.

***

Run the cells below to experiment with different queries. You can also modify the queries or add new cells.

***

[View the complete code](../financial_data_load/solution_srcs/06_03_graph_enriched_provider.py)

[Move on to the Fulltext Context Provider Notebook (Optional)](03_fulltext_context_provider.ipynb)

In [15]:
await ask("Tell me about Microsoft's cloud services and business risks")

User: Tell me about Microsoft's cloud services and business risks

Assistant: Microsoft Corporation (MSFT) has a robust suite of cloud services primarily delivered through its Azure platform. Here’s an overview of its cloud offerings and associated business risks:

### Cloud Services:

1. **Microsoft Azure**: A comprehensive cloud computing platform offering services such as virtual machines, app services, databases, and AI capabilities. It supports various programming languages and frameworks, making it versatile for developers.

2. **Microsoft 365**: A suite of productivity tools that combine Office applications (like Word, Excel, and PowerPoint) with cloud-based services such as OneDrive and SharePoint. It enables collaboration and productivity across devices.

3. **Dynamics 365**: A set of intelligent business applications for customer relationship management (CRM) and enterprise resource planning (ERP), blending data, AI, and productivity tools.

4. **Power Platform**: Comprising 

In [16]:
await ask("What products and risks are mentioned in Amazon's filings?")

User: What products and risks are mentioned in Amazon's filings?

Assistant: In Amazon.com, Inc.'s filings, the following products and risks are highlighted:

### Products:
1. **App Store**: Platform for distributing applications.
2. **Multifactor Authentication**: Security method to enhance account protection.
3. **Cellular-enabled Products**: Devices with cellular connectivity features.
4. **Search Services Licensing Arrangements**: Agreements for providing search capabilities to partners.
5. **iPhone**: Though primarily associated with Apple, it may refer to Amazon's compatibility or offerings related to the device.

### Risks:
1. **Component Supply Shortages**: Amazon may face disruptions in the availability of essential components needed for products.
2. **Commodity Price Fluctuations**: Variability in commodity prices can impact operational costs.
3. **Single-source/Custom Components Dependency**: Relying on specific suppliers for unique components could pose operational risks.
4

In [17]:
# Cleanup
await provider.__aexit__(None, None, None)
await credential.close()
embedder.close()